# Qwen3-4B × A-balanced (4M/500, micro-batch 4, max-seq 640)


In [ ]:
import json, shutil, subprocess, sys, os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:256'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['OMP_NUM_THREADS'] = '8'
os.environ['MKL_NUM_THREADS'] = '8'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'
from pathlib import Path
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('no GPU')
p = torch.cuda.get_device_properties(0)
print('gpu', p.name, round(p.total_memory / 1e9, 1), 'GB')
print('n_gpu', torch.cuda.device_count())
print('n_gpu', torch.cuda.device_count())
print('n_gpu', torch.cuda.device_count())
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], capture_output=True)


In [ ]:
DATA_SRC = Path('/kaggle/input/tiki-matrix-data')
if not (DATA_SRC / 'units_A.parquet').exists():
    hits = sorted(Path('/kaggle/input').rglob('units_A.parquet'))
    assert hits, 'không thấy tiki-matrix-data'
    DATA_SRC = hits[0].parent
ROOT = Path('/kaggle/working/matrix_data')
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
for name in ('A', 'B'):
    (ROOT / f'processed_{name}').mkdir(parents=True)
    shutil.copy2(DATA_SRC / f'units_{name}.parquet', ROOT / f'processed_{name}/units.parquet')
    shutil.copy2(DATA_SRC / f'eval_{name}.parquet', ROOT / f'eval_{name}.parquet')
    shutil.copy2(DATA_SRC / f'meta_{name}.json', ROOT / f'meta_{name}.json')
CODE = Path('/kaggle/working/code')
CODE.mkdir(exist_ok=True)
for name in ('train_weight_quota_gpu.py', 'quota_sampling.py', 'tpu_weight_train.py'):
    src = DATA_SRC / name
    if src.exists():
        shutil.copy2(src, CODE / name)

HIDDEN = 2560
MODEL = None
for cfg in Path('/kaggle/input').rglob('config.json'):
    try:
        d = json.loads(cfg.read_text(encoding='utf-8'))
    except Exception:
        continue
    if d.get('hidden_size') == HIDDEN and d.get('num_hidden_layers') in (36,):
        MODEL = str(cfg.parent)
        break
assert MODEL, f'không thấy model hidden={HIDDEN}'
print('DATA', DATA_SRC, '| MODEL', MODEL)


In [ ]:

import hashlib, re, unicodedata
from pathlib import Path
import pandas as pd
def _n(x):
    x = unicodedata.normalize("NFKC", str(x or "")).casefold()
    return re.sub(r"\\s+", " ", x).strip()
def _h(x):
    return hashlib.sha256(x.encode("utf-8")).hexdigest()
_hits = sorted(Path("/kaggle/input").rglob("eval_F_v2_common.parquet"))
assert _hits, "không thấy eval_F_v2_common.parquet"
v2 = pd.read_parquet(_hits[0])
ev_ids = set(pd.read_parquet(DATA_SRC / "eval_A.parquet").eval_id.astype(str))
assert set(v2.eval_id.astype(str)) <= ev_ids, "eval_F_v2_common không phải tập con của eval dùng để chấm"
for ds in ("A", "B"):
    U = pd.read_parquet(DATA_SRC / f"units_{ds}.parquet")
    tr = U[U.split.isin(["train", "dev"])]
    assert not (set(v2.product_id.astype(str)) & set(tr.product_id.astype(str))), f"LEAK product_id {ds}"
    ev_src = {_h(_n(x)) for x in v2.source_text}
    tr_src = {_h(_n(x)) for x in tr.source_text}
    assert not (ev_src & tr_src), f"LEAK source_hash {ds}: {len(ev_src & tr_src)}"
    ev_pair = {_h(_n(s) + "\\x1f" + _n(t)) for s, t in zip(v2.source_text, v2.target_text)}
    tr_pair = {_h(_n(s) + "\\x1f" + _n(t)) for s, t in zip(tr.source_text, tr.target_text)}
    assert not (ev_pair & tr_pair), f"LEAK pair_hash {ds}: {len(ev_pair & tr_pair)}"
print("PREFLIGHT 4-KEY LEAK CHECK OK", len(v2))


In [ ]:
import pandas as pd
ev = pd.read_parquet(DATA_SRC / 'eval_A.parquet')
for ds in ('A', 'B'):
    U = pd.read_parquet(DATA_SRC / f'units_{ds}.parquet')
    overlap = set(ev.product_id) & set(U.product_id)
    tok = float(U.eff_tokens.sum())
    print(ds, 'rows', len(U), 'tokensM', round(tok/1e6, 3),
          'overlap', len(overlap), 'epochs@4M', round(4e6/tok, 3))
    assert not overlap, f'LEAK {ds}: {len(overlap)} products'
print('LEAK CHECK OK')


In [ ]:
DATASET = 'B'
CONFIG = 'natural'
MODEL_LABEL = 'qwen4b'
OUT = Path('/kaggle/working/out_B_qwen4b_' + CONFIG)
if OUT.exists():
    shutil.rmtree(OUT)

# L4x4: 4x24GB nhưng script dùng 1 GPU -> 24GB. Thử batch 4, tự lùi 2 rồi 1 nếu OOM.
ok = False
for mb in (4, 2, 1):
    if OUT.exists():
        shutil.rmtree(OUT)
    cmd = [sys.executable, str(CODE / 'train_weight_quota_gpu.py'),
           '--dataset', DATASET, '--data-root', str(ROOT), '--out', str(OUT),
           '--model', MODEL, '--seed', '42', '--tokens', '4000000', '--steps', '500',
           '--configs', CONFIG, '--micro-batch', str(mb), '--max-seq', '960', '--dtype', 'fp16']
    print('RUN micro-batch', mb, flush=True)
    rc = subprocess.run(cmd).returncode
    print('rc', rc, flush=True)
    if rc == 0:
        ok = True
        break
if not ok:
    raise SystemExit('train failed for all micro-batch settings')

src = OUT / f'hyp_{CONFIG}_seed42.parquet'
dst = OUT / f'hyp_{MODEL_LABEL}_{DATASET.lower()}_{CONFIG}_seed42.parquet'
src.rename(dst)
print('HYP', dst, dst.stat().st_size)
print((OUT / 'summary.json').read_text())
